# 🧹 ml_preprocessor — End-to-End Example

This notebook demonstrates the full workflow of the `ml_preprocessor` framework:
1. Generate a synthetic HR dataset with real-world issues
2. Configure a pipeline via Python API
3. Fit on train, transform test
4. Generate an HTML preprocessing report
5. (Optional) Wrap a transformer for use in a sklearn Pipeline

---

In [ ]:
import sys
sys.path.insert(0, '..')  # adjust if running from notebooks/ subfolder

import numpy as np
import pandas as pd

from ml_preprocessor import (
    PreprocessingPipeline,
    MissingValueHandler,
    CategoricalEncoder,
    FeatureScaler,
    FeatureEngineer,
)
from ml_preprocessor.reporter import generate_report

## 1. Generate synthetic dataset

In [ ]:
np.random.seed(42)
N = 500

df = pd.DataFrame({
    'age':         np.random.randint(22, 62, N).astype(float),
    'salary':      np.random.normal(48_000, 18_000, N).round(2),
    'experience':  np.random.randint(0, 35, N).astype(float),
    'city':        np.random.choice(['Paris', 'Lyon', 'Lille', 'Bordeaux', None], N, p=[.35,.2,.2,.15,.1]),
    'department':  np.random.choice(['Tech', 'Sales', 'HR', 'Finance', 'Marketing'], N),
    'seniority':   np.random.choice(['intern', 'junior', 'mid', 'senior', 'lead'], N),
    'hire_date':   pd.date_range('2010-01-01', periods=N, freq='3D').astype(str),
    'perf_score':  np.random.uniform(1, 5, N).round(1),
})

# Inject missing values
for col, pct in [('age', .08), ('salary', .06), ('experience', .05), ('perf_score', .12)]:
    idx = np.random.choice(N, int(N * pct), replace=False)
    df.loc[idx, col] = np.nan

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Quick missing value overview
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

## 2. Train / Test split

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')

## 3. Build and run the pipeline

In [ ]:
pipeline = PreprocessingPipeline([

    # Step 1 — Impute numeric columns
    MissingValueHandler(
        columns=['age', 'salary', 'experience', 'perf_score'],
        strategy='median'
    ),

    # Step 2 — Fill missing city
    MissingValueHandler(columns=['city'], strategy='most_frequent'),

    # Step 3 — Feature engineering
    FeatureEngineer(
        interactions=[['salary', 'experience']],
        binning=[{
            'column': 'age',
            'bins': [0, 30, 40, 55, 100],
            'labels': ['young', 'mid', 'experienced', 'senior'],
            'drop_original': False,
        }],
        dates=[{
            'column': 'hire_date',
            'extract': ['year', 'month', 'dayofweek', 'is_weekend'],
            'drop_original': True,
        }],
        aggregations=[{
            'group_by': 'department',
            'agg_col': 'salary',
            'func': ['mean', 'std'],
            'prefix': 'dept_salary',
        }]
    ),

    # Step 4 — Encode ordinal
    CategoricalEncoder(
        columns=['seniority'],
        method='ordinal',
        ordinal_order={'seniority': ['intern', 'junior', 'mid', 'senior', 'lead']}
    ),

    # Step 5 — One-Hot encode nominals
    CategoricalEncoder(
        columns=['city', 'department'],
        method='onehot',
        drop_first=True
    ),

    # Step 6 — Scale numerics
    FeatureScaler(
        columns=['age', 'salary', 'experience', 'salary_x_experience', 'perf_score'],
        method='robust'
    ),

])

print(pipeline.summary())

In [ ]:
# Keep raw copy for the report
df_train_raw = df_train.copy()

df_train_clean = pipeline.fit_transform(df_train)
df_test_clean  = pipeline.transform(df_test)

print(f'Train clean: {df_train_clean.shape}')
print(f'Test clean:  {df_test_clean.shape}')
df_train_clean.head()

In [ ]:
# Verify no missing values remain
print('Missing values in cleaned train:')
print(df_train_clean.isnull().sum()[df_train_clean.isnull().sum() > 0])
print('✅ No missing values!' if df_train_clean.isnull().sum().sum() == 0 else '⚠️  Some missing remain')

## 4. Generate HTML report

In [ ]:
report_path = generate_report(
    df_before=df_train_raw,
    df_after=df_train_clean,
    output_path='preprocessing_report.html',
    title='HR Dataset'
)
print(f'Report generated: {report_path}')

# Render inline in notebook
from IPython.display import IFrame
IFrame(src=str(report_path), width='100%', height=700)

## 5. Save & reload pipeline

In [ ]:
pipeline.save('fitted_pipeline.pkl')

pipeline2 = PreprocessingPipeline.load('fitted_pipeline.pkl')
df_new = pipeline2.transform(df_test)
print('Reloaded pipeline — transform OK:', df_new.shape)

## 6. (Optional) sklearn integration

Any transformer can be wrapped into a sklearn-compatible object for use in `sklearn.pipeline.Pipeline`.

In [ ]:
try:
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import Ridge

    scaler = FeatureScaler(method='standard')

    sk_pipeline = Pipeline([
        ('scaler', scaler.to_sklearn()),
        ('model',  Ridge()),
    ])
    print('sklearn Pipeline built:', sk_pipeline)
except ImportError:
    print('scikit-learn not installed — skipping this cell')

---
## 7. CLI equivalent

Everything above can also be triggered from the command line:

```bash
# Run pipeline + generate report
python -m ml_preprocessor run \
  --config config_example.yaml \
  --input data/train.csv \
  --output data/train_clean.csv \
  --test-input data/test.csv \
  --save-pipeline pipeline.pkl \
  --report report.html

# Validate config before running
python -m ml_preprocessor validate --config config_example.yaml

# Print config schema
python -m ml_preprocessor schema
```